# Notebook 6 — URL Analysis Engine
### Fake Internship & Job Scam Detection System

**Purpose:**  
This notebook implements a full-stack URL Analysis Engine that accepts any job posting URL as input and produces a structured scam risk report by combining domain intelligence, web scraping, and the Detection Engine from Notebook 5.

**Pipeline Overview:**

| Step | Operation | Output |
|------|-----------|--------|
| 1  | Import Libraries | — |
| 2  | URL Validation | `is_valid_url` |
| 3  | Domain Extraction | `domain`, `subdomain`, `suffix`, `full_domain` |
| 4  | HTTPS Analysis | `https_enabled` |
| 5  | Domain Age (WHOIS) | `domain_age_days`, `creation_date` |
| 6  | Suspicious TLD Detection | `suspicious_tld` |
| 7  | Website Availability | `status_code`, `website_reachable` |
| 8  | Web Scraping | `page_title`, `body_text`, `emails`, `phone_numbers` |
| 9  | Content Cleaning | `clean_scraped_text` |
| 10 | Detection Engine Integration | `risk_score`, `risk_level`, `fraud_reasons` |
| 11 | Domain Risk Score | `domain_risk_score` |
| 12 | Final URL Risk Score | `final_url_risk_score` |
| 13 | Explainability Layer | `url_risk_reasons` |
| 14 | JSON Output — `analyze_url()` | Full structured report |
| 15 | Test Cases | Known / Suspicious / Invalid URL |
| 16 | Export Results | `url_analysis_results.csv` |


In [ ]:
# STEP 1: IMPORT LIBRARIES

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import re
import json
import time
from datetime import datetime, timezone
from urllib.parse import urlparse, urljoin

# --- HTTP & Scraping ---
try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'requests', 'beautifulsoup4', '-q'])
    import requests
    from bs4 import BeautifulSoup

# --- Domain Extraction ---
try:
    import tldextract
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tldextract', '-q'])
    import tldextract

# --- WHOIS ---
try:
    import whois
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-whois', '-q'])
    import whois

# --- Request settings ---
REQUEST_TIMEOUT = 10   # seconds
REQUEST_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

print("All libraries loaded successfully.")
print(f"  requests      : {requests.__version__}")
print(f"  tldextract    : {tldextract.__version__}")
print(f"  beautifulsoup4: installed")
print(f"  python-whois  : installed")

In [ ]:
# STEP 2: URL VALIDATION

def validate_url(url: str) -> bool:
    """
    Validate that the input string is a well-formed HTTP/HTTPS URL.

    Parameters
    ----------
    url : str
        Raw URL string to validate.

    Returns
    -------
    bool
        True if the URL is structurally valid, False otherwise.
    """
    if not isinstance(url, str) or not url.strip():
        return False
    try:
        parsed = urlparse(url.strip())
        # Scheme must be http or https
        if parsed.scheme not in ('http', 'https'):
            return False
        # Network location (domain) must be present
        if not parsed.netloc:
            return False
        # Domain must contain at least one dot
        if '.' not in parsed.netloc:
            return False
        return True
    except Exception:
        return False


# --- Unit tests ---
test_cases = [
    ('https://www.google.com',              True),
    ('http://careers-company.xyz/job123',   True),
    ('ftp://files.example.com',             False),   # invalid scheme
    ('not-a-url',                           False),   # no scheme
    ('https://',                            False),   # no domain
    ('',                                    False),   # empty string
    ('https://localhost',                   False),   # no dot in domain
]

print("URL Validation Unit Tests:")
print(f"{'URL':<45} {'Expected':<10} {'Result':<10} {'Pass?'}")
print('-' * 75)
all_passed = True
for url, expected in test_cases:
    result = validate_url(url)
    passed = '✓' if result == expected else '✗ FAIL'
    if result != expected:
        all_passed = False
    print(f"{url:<45} {str(expected):<10} {str(result):<10} {passed}")

print(f"\nAll tests passed: {all_passed}")

In [ ]:
# STEP 3: EXTRACT DOMAIN INFORMATION

def extract_domain_info(url: str) -> dict:
    """
    Decompose a URL into subdomain, domain, suffix and full_domain.

    Returns a dict with keys: subdomain, domain, suffix, full_domain.
    Returns empty strings on failure.
    """
    try:
        extracted   = tldextract.extract(url)
        subdomain   = extracted.subdomain  or ''
        domain      = extracted.domain    or ''
        suffix      = extracted.suffix    or ''
        # Build clean full_domain — omit empty subdomain
        parts       = [p for p in [subdomain, domain, suffix] if p]
        full_domain = '.'.join(parts)
        return {
            'subdomain'  : subdomain,
            'domain'     : domain,
            'suffix'     : suffix,
            'full_domain': full_domain
        }
    except Exception as e:
        return {'subdomain': '', 'domain': '', 'suffix': '', 'full_domain': ''}


# --- Demonstration ---
demo_urls = [
    'https://abc.jobs.xyz/apply?id=123',
    'https://careers.infosys.com/job/developer',
    'http://earn-money-fast.work/register',
    'https://internship.co.in/listing/456',
]

print("Domain Extraction Examples:")
print(f"{'URL':<45} {'subdomain':<12} {'domain':<15} {'suffix':<8} {'full_domain'}")
print('-' * 100)
for u in demo_urls:
    info = extract_domain_info(u)
    print(f"{u:<45} {info['subdomain']:<12} {info['domain']:<15} "
          f"{info['suffix']:<8} {info['full_domain']}")

In [ ]:
# STEP 4: HTTPS ANALYSIS

def check_https(url: str) -> bool:
    """
    Return True if the URL uses HTTPS, False if HTTP.
    """
    try:
        return urlparse(url.strip()).scheme.lower() == 'https'
    except Exception:
        return False


# --- Demonstration ---
https_tests = [
    'https://careers.google.com/jobs',
    'http://earn-money-now.xyz/apply',
    'https://internship.co.in/register',
    'http://register-now.work/fee',
]

print("HTTPS Analysis:")
for u in https_tests:
    result = check_https(u)
    status = 'SECURE (HTTPS)' if result else 'SUSPICIOUS (HTTP only)'
    print(f"  {u:<45}  →  {status}")

In [ ]:
# STEP 5: DOMAIN AGE ANALYSIS (WHOIS)

def get_domain_age(url: str) -> dict:
    """
    Perform WHOIS lookup and return domain registration metadata.

    Returns
    -------
    dict with keys:
        creation_date    : str  (ISO format) or 'Unknown'
        domain_age_days  : int  or -1 if lookup failed
        domain_age_risk  : str  'HIGH' | 'MEDIUM' | 'LOW' | 'UNKNOWN'
    """
    domain_info = extract_domain_info(url)
    root_domain = f"{domain_info['domain']}.{domain_info['suffix']}"

    try:
        w = whois.whois(root_domain)
        creation = w.creation_date

        # creation_date can be a list — take the earliest
        if isinstance(creation, list):
            creation = min(
                [d for d in creation if isinstance(d, datetime)],
                default=None
            )

        if creation is None or not isinstance(creation, datetime):
            return {'creation_date': 'Unknown',
                    'domain_age_days': -1,
                    'domain_age_risk': 'UNKNOWN'}

        # Normalise timezone — strip tz info for naive comparison
        now = datetime.now()
        if creation.tzinfo is not None:
            creation = creation.replace(tzinfo=None)

        age_days = (now - creation).days

        if age_days < 30:
            risk = 'HIGH'
        elif age_days < 180:
            risk = 'MEDIUM'
        else:
            risk = 'LOW'

        return {
            'creation_date'  : creation.strftime('%Y-%m-%d'),
            'domain_age_days': age_days,
            'domain_age_risk': risk
        }

    except Exception as e:
        return {'creation_date': 'Unknown',
                'domain_age_days': -1,
                'domain_age_risk': 'UNKNOWN'}


print("get_domain_age() function defined.")
print("Risk thresholds:")
print("  < 30 days  → HIGH RISK")
print("  30-180 days → MEDIUM RISK")
print("  > 180 days  → LOW RISK")
print("  WHOIS failure → UNKNOWN (safe default, does not crash pipeline)")

In [ ]:
# STEP 6: SUSPICIOUS TLD DETECTION

SUSPICIOUS_TLDS = {
    'xyz', 'top', 'click', 'online', 'site', 'live',
    'shop', 'buzz', 'work', 'loan', 'win', 'gq',
    'ml', 'cf', 'tk', 'ga', 'pw', 'cc', 'biz',
    'info', 'mobi', 'name', 'pro', 'link', 'space',
    'website', 'press', 'rocks', 'fun', 'icu'
}

def check_suspicious_tld(url: str) -> dict:
    """
    Check whether the URL's TLD appears in the suspicious TLD list.

    Returns
    -------
    dict with keys:
        suffix          : str  — actual TLD found
        suspicious_tld  : int  — 1 if suspicious, 0 if safe
    """
    domain_info = extract_domain_info(url)
    # Take only the final component for multi-part TLDs (e.g. 'co.uk' → 'uk')
    suffix      = domain_info.get('suffix', '').lower()
    final_tld   = suffix.split('.')[-1] if suffix else ''
    is_suspicious = 1 if final_tld in SUSPICIOUS_TLDS else 0
    return {
        'suffix'        : suffix,
        'suspicious_tld': is_suspicious
    }


# --- Demonstration ---
tld_tests = [
    'https://careers.infosys.com/jobs',
    'https://jobs.earn-fast.xyz/apply',
    'http://register-now.work/fee',
    'https://internship.co.in/apply',
    'https://hiring.buzz/data-entry',
    'https://google.com/careers',
]

print("Suspicious TLD Detection:")
print(f"{'URL':<48} {'TLD':<8} {'Suspicious?'}")
print('-' * 70)
for u in tld_tests:
    result = check_suspicious_tld(u)
    flag   = 'YES ⚠' if result['suspicious_tld'] else 'No'
    print(f"{u:<48} {result['suffix']:<8} {flag}")

In [ ]:
# STEP 7: WEBSITE AVAILABILITY CHECK

def check_website_availability(url: str) -> dict:
    """
    Attempt an HTTP GET to the URL and return availability info.

    Returns
    -------
    dict with keys:
        status_code       : int   — HTTP status or -1 on failure
        website_reachable : bool  — True if status 200-399
        response_time_ms  : float — round-trip time in ms
        error_message     : str   — '' on success, error text on failure
    """
    try:
        start    = time.time()
        response = requests.get(
            url,
            headers = REQUEST_HEADERS,
            timeout = REQUEST_TIMEOUT,
            allow_redirects = True
        )
        elapsed  = round((time.time() - start) * 1000, 2)
        reachable = 200 <= response.status_code < 400
        return {
            'status_code'      : response.status_code,
            'website_reachable': reachable,
            'response_time_ms' : elapsed,
            'error_message'    : ''
        }
    except requests.exceptions.SSLError:
        return {'status_code': -1, 'website_reachable': False,
                'response_time_ms': 0, 'error_message': 'SSL certificate error'}
    except requests.exceptions.ConnectionError:
        return {'status_code': -1, 'website_reachable': False,
                'response_time_ms': 0, 'error_message': 'Connection refused or DNS failure'}
    except requests.exceptions.Timeout:
        return {'status_code': -1, 'website_reachable': False,
                'response_time_ms': 0, 'error_message': f'Request timed out after {REQUEST_TIMEOUT}s'}
    except Exception as e:
        return {'status_code': -1, 'website_reachable': False,
                'response_time_ms': 0, 'error_message': str(e)}


print("check_website_availability() function defined.")
print(f"  Timeout setting : {REQUEST_TIMEOUT} seconds")
print("  Reachable if    : HTTP status 200 – 399")
print("  Safe defaults   : status=-1, reachable=False on any exception")

In [ ]:
# STEP 8: WEB SCRAPING

def scrape_url(url: str) -> dict:
    """
    Fetch a URL and extract structured content using BeautifulSoup.

    Returns
    -------
    dict with keys:
        page_title, meta_description, body_text,
        all_links, emails, phone_numbers, scrape_success
    """
    default = {
        'page_title'      : '',
        'meta_description': '',
        'body_text'       : '',
        'all_links'       : [],
        'emails'          : [],
        'phone_numbers'   : [],
        'scrape_success'  : False
    }

    try:
        response = requests.get(
            url,
            headers = REQUEST_HEADERS,
            timeout = REQUEST_TIMEOUT,
            allow_redirects = True
        )
        if response.status_code != 200:
            default['scrape_success'] = False
            return default

        soup = BeautifulSoup(response.text, 'html.parser')

        # --- Page title ---
        title_tag  = soup.find('title')
        page_title = title_tag.get_text(strip=True) if title_tag else ''

        # --- Meta description ---
        meta_tag   = soup.find('meta', attrs={'name': re.compile('description', re.I)})
        meta_desc  = meta_tag.get('content', '') if meta_tag else ''

        # --- Body text (remove scripts and styles first) ---
        for tag in soup(['script', 'style', 'noscript', 'header', 'footer', 'nav']):
            tag.decompose()
        body_text  = soup.get_text(separator=' ', strip=True)

        # --- All hyperlinks ---
        all_links  = list(set(
            a.get('href', '') for a in soup.find_all('a', href=True)
            if a.get('href', '').startswith('http')
        ))

        # --- Email addresses ---
        email_pattern = r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}'
        emails        = list(set(re.findall(email_pattern, response.text)))

        # --- Phone numbers (Indian + international formats) ---
        phone_pattern  = r'(?:\+91[\-\s]?)?[6-9]\d{9}|\b\d{7,12}\b'
        phone_numbers  = list(set(re.findall(phone_pattern, body_text)))

        return {
            'page_title'      : page_title,
            'meta_description': meta_desc,
            'body_text'       : body_text,
            'all_links'       : all_links,
            'emails'          : emails,
            'phone_numbers'   : phone_numbers,
            'scrape_success'  : True
        }

    except Exception as e:
        default['scrape_success'] = False
        return default


print("scrape_url() function defined.")
print("Extracted fields: page_title | meta_description | body_text")
print("                  all_links  | emails           | phone_numbers")

In [ ]:
# STEP 9: CONTENT CLEANING

MAX_CLEAN_CHARS = 5000

def clean_scraped_text(raw_text: str) -> str:
    """
    Normalise raw scraped body text for Detection Engine input.

    Parameters
    ----------
    raw_text : str  — raw body_text from scrape_url()

    Returns
    -------
    str — cleaned, lowercase, truncated text
    """
    if not raw_text or not isinstance(raw_text, str):
        return ''
    text = raw_text.lower()
    text = re.sub(r'https?://\S+|www\.\S+',        ' ', text)   # URLs
    text = re.sub(r'\S+@\S+\.\S+',                 ' ', text)   # Emails
    text = re.sub(r'[^a-z\s]',                     ' ', text)   # Non-alpha
    text = re.sub(r'\s+',                           ' ', text)   # Whitespace
    text = text.strip()
    return text[:MAX_CLEAN_CHARS]


# --- Demonstration ---
sample_raw = """
    <p>URGENT HIRING!! Work From Home Job — Earn ₹50,000/month!
    No experience needed. WhatsApp us at +91-9876543210.
    Visit: http://earn-fast.xyz/register — Pay ₹500 registration fee.
    Contact: scamjobs@gmail.com. Limited seats available!!
"""
cleaned = clean_scraped_text(sample_raw)
print("Content Cleaning Demo:")
print(f"  Raw length    : {len(sample_raw)} chars")
print(f"  Cleaned length: {len(cleaned)} chars")
print(f"  Cleaned text  : {cleaned}")

In [ ]:
# STEP 10: DETECTION ENGINE INTEGRATION

from textstat import flesch_reading_ease

# --- Weighted fraud phrase dictionary (Notebook 3 source) ---
ENGINE_FRAUD_PHRASES = {
    'registration fee' : 15, 'security deposit' : 15,
    'processing fee'   : 12, 'training fee'     : 12,
    'pay to work'      : 15, 'investment'       : 12,
    'earn online'      : 10, 'earn money fast'  : 12,
    'daily payment'    : 10, 'income source'    :  8,
    'form filling'     :  8, 'typing job'       :  8,
    'data entry'       :  6, 'copy paste'       : 10,
    'captcha'          :  8, 'ad posting'       : 10,
    'refer and earn'   : 10, 'multi level'      : 12,
    'refundable deposit': 14,'no experience needed': 6,
}

ENGINE_URGENCY_PHRASES = [
    'urgent', 'immediately', 'limited seats', 'limited openings',
    'hurry', 'apply now', 'last date', 'closing soon',
    'act fast', 'do not miss', 'first come first', 'fast track'
]

ENGINE_CONTACT_RISKS = {
    'whatsapp': 10, 'telegram': 10, 'gmail.com': 6,
    'yahoo.com': 6, 'call now': 8,  'call us': 5,
    'dm us': 6,     'direct message': 6, 'ping us': 5,
}


def analyze_job_text(text: str) -> dict:
    """
    Run the Detection Engine on a cleaned job description text.

    Returns
    -------
    dict with keys:
        content_risk_score, risk_level, fraud_reasons,
        recommended_action, fraud_phrase_score,
        urgency_score, contact_risk_score, readability_score
    """
    reasons = []
    t       = text.lower()

    # --- Fraud phrase scoring ---
    phrase_score = sum(
        w for p, w in ENGINE_FRAUD_PHRASES.items() if p in t
    )
    for p, w in ENGINE_FRAUD_PHRASES.items():
        if p in t:
            reasons.append(f"Fraud phrase detected: '{p}' (weight {w})")

    # --- Urgency scoring ---
    urgency_score = sum(1 for p in ENGINE_URGENCY_PHRASES if p in t)
    if urgency_score > 0:
        reasons.append(f"Urgency language detected ({urgency_score} cues)")

    # --- Contact risk scoring ---
    contact_score = sum(
        w for c, w in ENGINE_CONTACT_RISKS.items() if c in t
    )
    for c, w in ENGINE_CONTACT_RISKS.items():
        if c in t:
            reasons.append(f"Risky contact channel: '{c}' (weight {w})")

    # --- Readability scoring ---
    try:
        readability = flesch_reading_ease(text)
        if readability > 85:
            reasons.append(f"Very high readability score ({readability:.1f}) — unusually simple language")
    except Exception:
        readability = 50.0

    # --- Composite content risk score (cap at 100) ---
    raw_score = phrase_score + (urgency_score * 5) + contact_score
    content_risk_score = min(int(raw_score), 100)

    # --- Risk level bands ---
    if content_risk_score >= 60:
        risk_level = 'CRITICAL'
        action     = 'DO NOT APPLY — High probability scam. Report immediately.'
    elif content_risk_score >= 35:
        risk_level = 'HIGH'
        action     = 'Proceed with extreme caution. Verify company independently.'
    elif content_risk_score >= 15:
        risk_level = 'MEDIUM'
        action     = 'Exercise caution. Research the company before applying.'
    else:
        risk_level = 'LOW'
        action     = 'Appears relatively safe. Standard due diligence recommended.'

    return {
        'content_risk_score' : content_risk_score,
        'risk_level'         : risk_level,
        'fraud_reasons'      : reasons,
        'recommended_action' : action,
        'fraud_phrase_score' : phrase_score,
        'urgency_score'      : urgency_score,
        'contact_risk_score' : contact_score,
        'readability_score'  : round(readability, 2)
    }


# --- Quick engine test ---
test_text = clean_scraped_text(
    'URGENT!! Work from home job. Pay registration fee of Rs 500. '
    'Earn online daily. Contact us on whatsapp. Limited seats available. '
    'No experience needed. Daily payment guaranteed.'
)
engine_result = analyze_job_text(test_text)
print("Detection Engine — Quick Test:")
print(f"  content_risk_score : {engine_result['content_risk_score']}")
print(f"  risk_level         : {engine_result['risk_level']}")
print(f"  fraud_reasons      :")
for r in engine_result['fraud_reasons']:
    print(f"    • {r}")

In [ ]:
# ============================================================
# STEP 11: DOMAIN RISK SCORE
# Combine the four domain-level signals into a single numeric
# domain_risk_score on a 0–100 scale.
#
# Weighted components:
#
#   Signal               Max Points  Rationale
#   ─────────────────    ──────────  ──────────────────────────────
#   Domain age           40 pts      Newly registered = highest risk
#   HTTPS                20 pts      HTTP-only in 2024 is suspicious
#   Suspicious TLD       25 pts      .xyz/.work/.buzz etc.
#   Website reachability 15 pts      Offline sites are red flags
#
# Domain age scoring:
#   < 30 days    → 40 pts (HIGH)
#   30–180 days  → 20 pts (MEDIUM)
#   > 180 days   →  0 pts (LOW)
#   UNKNOWN      → 15 pts (partial penalty)
# ============================================================

def compute_domain_risk_score(https_enabled: bool,
                               domain_age_days: int,
                               domain_age_risk: str,
                               suspicious_tld: int,
                               website_reachable: bool) -> dict:
    """
    Compute a weighted domain risk score from signal inputs.

    Returns
    -------
    dict with keys:
        domain_risk_score   : int  (0–100)
        domain_risk_level   : str  'LOW' | 'MEDIUM' | 'HIGH'
        domain_risk_breakdown : dict of component scores
    """
    breakdown = {}

    # --- Domain age component (max 40) ---
    if domain_age_risk == 'HIGH':
        breakdown['domain_age_score'] = 40
    elif domain_age_risk == 'MEDIUM':
        breakdown['domain_age_score'] = 20
    elif domain_age_risk == 'LOW':
        breakdown['domain_age_score'] = 0
    else:  # UNKNOWN
        breakdown['domain_age_score'] = 15

    # --- HTTPS component (max 20) ---
    breakdown['https_score'] = 0 if https_enabled else 20

    # --- Suspicious TLD component (max 25) ---
    breakdown['tld_score'] = 25 if suspicious_tld else 0

    # --- Website availability component (max 15) ---
    breakdown['availability_score'] = 0 if website_reachable else 15

    domain_risk_score = sum(breakdown.values())
    domain_risk_score = min(domain_risk_score, 100)

    if domain_risk_score >= 60:
        domain_risk_level = 'HIGH'
    elif domain_risk_score >= 30:
        domain_risk_level = 'MEDIUM'
    else:
        domain_risk_level = 'LOW'

    return {
        'domain_risk_score'    : domain_risk_score,
        'domain_risk_level'    : domain_risk_level,
        'domain_risk_breakdown': breakdown
    }


# --- Quick test ---
demo_domain_risk = compute_domain_risk_score(
    https_enabled    = False,
    domain_age_days  = 12,
    domain_age_risk  = 'HIGH',
    suspicious_tld   = 1,
    website_reachable= True
)
print("Domain Risk Score Demo (HTTP, 12-day-old .xyz domain):")
for k, v in demo_domain_risk.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
# STEP 12: FINAL URL RISK SCORE
# Combine the domain-level risk score (Step 11) and the
# content-level risk score (Step 10) into a single unified
# final_url_risk_score on a 0–100 scale.
#
# Weighting rationale:
#   Domain risk  — 40% weight
#     Structural signals (age, TLD, HTTPS) are objective
#     and harder to fake than page content.
#
#   Content risk — 60% weight
#     The actual language of the job posting is the most
#     direct indicator of fraudulent intent.
#
# Formula:
#   final = (domain_risk_score × 0.4) + (content_risk_score × 0.6)
#   capped at 100.
#
# Final risk bands:
#   0–24   → LOW
#   25–49  → MEDIUM
#   50–74  → HIGH
#   75–100 → CRITICAL
# ============================================================

DOMAIN_WEIGHT  = 0.40
CONTENT_WEIGHT = 0.60

def compute_final_risk_score(domain_risk_score: int,
                              content_risk_score: int) -> dict:
    """
    Compute the final blended URL risk score.

    Returns
    -------
    dict with keys:
        final_url_risk_score : int  (0–100)
        final_risk_level     : str  'LOW' | 'MEDIUM' | 'HIGH' | 'CRITICAL'
    """
    raw = (domain_risk_score  * DOMAIN_WEIGHT +
           content_risk_score * CONTENT_WEIGHT)
    final_score = min(int(round(raw)), 100)

    if final_score >= 75:
        level = 'CRITICAL'
    elif final_score >= 50:
        level = 'HIGH'
    elif final_score >= 25:
        level = 'MEDIUM'
    else:
        level = 'LOW'

    return {
        'final_url_risk_score': final_score,
        'final_risk_level'    : level
    }


# --- Score band reference table ---
print("Final URL Risk Score — Band Reference:")
print("  0  – 24  → LOW       (likely legitimate)")
print("  25 – 49  → MEDIUM    (exercise caution)")
print("  50 – 74  → HIGH      (probable scam)")
print("  75 – 100 → CRITICAL  (do not apply)")
print(f"\n  Blending: domain × {DOMAIN_WEIGHT} + content × {CONTENT_WEIGHT}")

In [ ]:
# ============================================================
# STEP 13: EXPLAINABILITY LAYER
# Generate a consolidated, human-readable list of risk reasons
# combining domain-level and content-level signals.
#
# This layer is what makes the system actionable — instead of
# just a number, the user sees exactly WHY the URL was flagged.
#
# Domain-level reason triggers:
#   • Newly registered domain (< 30 days)
#   • Recently registered domain (30–180 days)
#   • Suspicious TLD detected
#   • Website unavailable or unreachable
#   • HTTP-only website (no HTTPS)
#   • Domain age unknown (WHOIS lookup failed)
#
# Content-level reasons come directly from analyze_job_text()
# (fraud phrases, urgency cues, risky contact channels).
# ============================================================

def build_risk_reasons(domain_age_days: int,
                        domain_age_risk: str,
                        suspicious_tld: int,
                        suffix: str,
                        https_enabled: bool,
                        website_reachable: bool,
                        content_fraud_reasons: list,
                        emails: list,
                        phone_numbers: list) -> list:
    """
    Consolidate all risk signals into a single url_risk_reasons list.

    Returns
    -------
    list of str — ordered from highest-severity to lowest
    """
    reasons = []

    # --- Domain age ---
    if domain_age_risk == 'HIGH' and domain_age_days >= 0:
        reasons.append(f"Newly registered domain — only {domain_age_days} days old (HIGH RISK)")
    elif domain_age_risk == 'MEDIUM' and domain_age_days >= 0:
        reasons.append(f"Recently registered domain — {domain_age_days} days old (MEDIUM RISK)")
    elif domain_age_risk == 'UNKNOWN':
        reasons.append("Domain age unknown — WHOIS lookup failed or domain is privacy-protected")

    # --- HTTPS ---
    if not https_enabled:
        reasons.append("Website uses HTTP only — no SSL/TLS encryption (suspicious in 2024)")

    # --- Suspicious TLD ---
    if suspicious_tld:
        reasons.append(f"Suspicious TLD detected: .{suffix} — commonly used by fraudulent sites")

    # --- Availability ---
    if not website_reachable:
        reasons.append("Website is currently unreachable — may have been taken down")

    # --- Contact channels ---
    if emails:
        free_email_domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'rediffmail.com']
        suspicious_emails  = [e for e in emails if any(d in e for d in free_email_domains)]
        if suspicious_emails:
            reasons.append(f"Recruiter uses free email service(s): {suspicious_emails[:3]}")

    if len(phone_numbers) > 2:
        reasons.append(f"Multiple phone numbers found ({len(phone_numbers)}) — unusual for legitimate postings")

    # --- Append content-level reasons from Detection Engine ---
    reasons.extend(content_fraud_reasons)

    return reasons


print("build_risk_reasons() function defined.")
print("Reason categories:")
print("  Domain   : age, HTTPS, TLD, availability")
print("  Contacts : suspicious email domains, multiple phones")
print("  Content  : fraud phrases, urgency cues, contact channels")

In [ ]:
# ============================================================
# STEP 14: MASTER FUNCTION — analyze_url()
# This is the single entry point for the entire URL Analysis
# Engine.  It chains all 13 preceding steps into one call and
# returns a fully structured JSON-serialisable report.
#
# Input  : A raw URL string
# Output : A dict (JSON report) containing:
#   • url, is_valid
#   • Domain info (subdomain, domain, suffix, full_domain)
#   • HTTPS, domain_age_days, creation_date
#   • suspicious_tld, status_code, website_reachable
#   • page_title, meta_description, emails, phone_numbers
#   • content_risk_score, domain_risk_score
#   • final_url_risk_score, final_risk_level
#   • url_risk_reasons (consolidated list)
#   • recommended_action
#   • analysis_timestamp
# ============================================================

def analyze_url(url: str,
                run_whois: bool = True,
                verbose: bool   = True) -> dict:
    """
    Full URL analysis pipeline — single entry point.

    Parameters
    ----------
    url       : str  — raw URL to analyse
    run_whois : bool — set False to skip WHOIS (faster, no domain age)
    verbose   : bool — print progress steps if True

    Returns
    -------
    dict — full JSON-serialisable risk report
    """

    if verbose: print(f"\n{'='*60}")
    if verbose: print(f"  Analysing: {url}")
    if verbose: print(f"{'='*60}")

    # ── Step 2: Validate ──────────────────────────────────────
    is_valid = validate_url(url)
    if not is_valid:
        if verbose: print("  [INVALID URL] Returning error report.")
        return {
            'url'                  : url,
            'is_valid'             : False,
            'final_url_risk_score' : 100,
            'final_risk_level'     : 'INVALID',
            'url_risk_reasons'     : ['URL is malformed or uses an unsupported scheme'],
            'recommended_action'   : 'This URL is not a valid HTTP/HTTPS address.',
            'analysis_timestamp'   : datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

    # ── Step 3: Domain extraction ─────────────────────────────
    if verbose: print("  [1/7] Extracting domain info...")
    domain_info = extract_domain_info(url)

    # ── Step 4: HTTPS ─────────────────────────────────────────
    https_enabled = check_https(url)

    # ── Step 5: WHOIS ─────────────────────────────────────────
    if verbose: print("  [2/7] WHOIS domain age lookup...")
    if run_whois:
        age_info = get_domain_age(url)
    else:
        age_info = {'creation_date': 'Skipped', 'domain_age_days': -1, 'domain_age_risk': 'UNKNOWN'}

    # ── Step 6: TLD check ─────────────────────────────────────
    if verbose: print("  [3/7] Checking TLD...")
    tld_info = check_suspicious_tld(url)

    # ── Step 7: Availability ──────────────────────────────────
    if verbose: print("  [4/7] Checking website availability...")
    avail_info = check_website_availability(url)

    # ── Steps 8 & 9: Scrape + Clean ───────────────────────────
    if verbose: print("  [5/7] Scraping page content...")
    if avail_info['website_reachable']:
        scrape_data = scrape_url(url)
        clean_text  = clean_scraped_text(scrape_data['body_text'])
    else:
        scrape_data = {
            'page_title': '', 'meta_description': '', 'body_text': '',
            'all_links': [], 'emails': [], 'phone_numbers': [],
            'scrape_success': False
        }
        clean_text = ''

    # ── Step 10: Detection Engine ─────────────────────────────
    if verbose: print("  [6/7] Running Detection Engine...")
    engine_result = analyze_job_text(clean_text)

    # ── Step 11: Domain risk score ────────────────────────────
    domain_risk = compute_domain_risk_score(
        https_enabled     = https_enabled,
        domain_age_days   = age_info['domain_age_days'],
        domain_age_risk   = age_info['domain_age_risk'],
        suspicious_tld    = tld_info['suspicious_tld'],
        website_reachable = avail_info['website_reachable']
    )

    # ── Step 12: Final URL risk score ─────────────────────────
    final_risk = compute_final_risk_score(
        domain_risk_score  = domain_risk['domain_risk_score'],
        content_risk_score = engine_result['content_risk_score']
    )

    # ── Step 13: Explainability ───────────────────────────────
    if verbose: print("  [7/7] Building risk reasons...")
    url_risk_reasons = build_risk_reasons(
        domain_age_days       = age_info['domain_age_days'],
        domain_age_risk       = age_info['domain_age_risk'],
        suspicious_tld        = tld_info['suspicious_tld'],
        suffix                = tld_info['suffix'],
        https_enabled         = https_enabled,
        website_reachable     = avail_info['website_reachable'],
        content_fraud_reasons = engine_result['fraud_reasons'],
        emails                = scrape_data['emails'],
        phone_numbers         = scrape_data['phone_numbers']
    )

    # ── Assemble final report ──────────────────────────────────
    report = {
        'url'                    : url,
        'is_valid'               : True,
        # Domain metadata
        'subdomain'              : domain_info['subdomain'],
        'domain'                 : domain_info['domain'],
        'suffix'                 : domain_info['suffix'],
        'full_domain'            : domain_info['full_domain'],
        # Security signals
        'https_enabled'          : https_enabled,
        'creation_date'          : age_info['creation_date'],
        'domain_age_days'        : age_info['domain_age_days'],
        'domain_age_risk'        : age_info['domain_age_risk'],
        'suspicious_tld'         : bool(tld_info['suspicious_tld']),
        # Availability
        'status_code'            : avail_info['status_code'],
        'website_reachable'      : avail_info['website_reachable'],
        'response_time_ms'       : avail_info['response_time_ms'],
        # Scraped content
        'page_title'             : scrape_data['page_title'],
        'meta_description'       : scrape_data['meta_description'],
        'emails_found'           : scrape_data['emails'],
        'phone_numbers_found'    : scrape_data['phone_numbers'],
        'links_found'            : len(scrape_data['all_links']),
        # Scores
        'fraud_phrase_score'     : engine_result['fraud_phrase_score'],
        'urgency_score'          : engine_result['urgency_score'],
        'contact_risk_score'     : engine_result['contact_risk_score'],
        'readability_score'      : engine_result['readability_score'],
        'content_risk_score'     : engine_result['content_risk_score'],
        'domain_risk_score'      : domain_risk['domain_risk_score'],
        'domain_risk_breakdown'  : domain_risk['domain_risk_breakdown'],
        'final_url_risk_score'   : final_risk['final_url_risk_score'],
        'final_risk_level'       : final_risk['final_risk_level'],
        # Explainability
        'url_risk_reasons'       : url_risk_reasons,
        'recommended_action'     : engine_result['recommended_action'],
        'analysis_timestamp'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    if verbose:
        print(f"\n  ✓ Analysis complete")
        print(f"  Final Risk Score : {report['final_url_risk_score']} / 100")
        print(f"  Risk Level       : {report['final_risk_level']}")
        print(f"  Reasons found    : {len(url_risk_reasons)}")

    return report


print("analyze_url() master function defined and ready.")

In [ ]:
# ============================================================
# STEP 15: TEST CASES
# Run the full analyze_url() pipeline on three representative
# test scenarios that cover the complete risk spectrum.
#
# Test 1 — Legitimate company URL
#   Expected: LOW risk, established domain, HTTPS, clean content
#
# Test 2 — Suspicious scam-pattern URL
#   Expected: HIGH/CRITICAL risk, suspicious TLD, scam content
#
# Test 3 — Invalid / malformed URL
#   Expected: INVALID response, pipeline exits gracefully
#
# Note: WHOIS is disabled (run_whois=False) for Test 2 and 3
# to avoid rate-limit delays on non-existent domains.
# Set run_whois=True in production use.
# ============================================================

TEST_URLS = [
    {
        'label': 'Test 1 — Legitimate Company URL',
        'url'  : 'https://www.infosys.com/careers/',
        'whois': True
    },
    {
        'label': 'Test 2 — Suspicious Scam-Pattern URL (simulated)',
        'url'  : 'http://earn-money-fast.xyz/register-now',
        'whois': False
    },
    {
        'label': 'Test 3 — Invalid / Malformed URL',
        'url'  : 'not-a-valid-url',
        'whois': False
    }
]

all_results = []

for test in TEST_URLS:
    print(f"\n{'#'*60}")
    print(f"  {test['label']}")
    print(f"{'#'*60}")

    result = analyze_url(test['url'], run_whois=test['whois'], verbose=True)
    all_results.append(result)

    # --- Pretty print the JSON report ---
    printable = {k: v for k, v in result.items()
                 if k not in ('domain_risk_breakdown',)}
    print("\nFull Report:")
    print(json.dumps(printable, indent=2, default=str))

In [ ]:
# --- Test Results Summary Table ---

print("\n" + "="*80)
print(" URL ANALYSIS ENGINE — TEST RESULTS SUMMARY")
print("="*80)
print(f"{'#':<4} {'URL':<42} {'Risk Score':<12} {'Level':<10} {'Reasons'}")
print('-'*80)

for i, result in enumerate(all_results, 1):
    url_short    = result['url'][:40] + '..' if len(result['url']) > 40 else result['url']
    risk_score   = result.get('final_url_risk_score', 'N/A')
    risk_level   = result.get('final_risk_level', 'N/A')
    reason_count = len(result.get('url_risk_reasons', []))
    print(f"{i:<4} {url_short:<42} {str(risk_score):<12} {risk_level:<10} {reason_count} reason(s)")

print("="*80)

In [ ]:
# ============================================================
# STEP 16: EXPORT RESULTS
# Flatten the structured result dicts from Step 15 into a
# tabular DataFrame and save to CSV.
#
# List columns are serialised to JSON strings for CSV
# compatibility (emails_found, url_risk_reasons, etc.).
#
# Output file: ../data/url_analysis_results.csv
# ============================================================

# --- Flatten nested/list fields for CSV export ---
def flatten_result_for_csv(result: dict) -> dict:
    """
    Prepare a result dict for CSV by serialising list/dict fields
    to JSON strings.
    """
    flat = {}
    list_dict_fields = {
        'emails_found', 'phone_numbers_found',
        'url_risk_reasons', 'domain_risk_breakdown'
    }
    for k, v in result.items():
        if k in list_dict_fields:
            flat[k] = json.dumps(v, default=str)
        else:
            flat[k] = v
    return flat


# --- Build DataFrame ---
flat_results = [flatten_result_for_csv(r) for r in all_results]
results_df   = pd.DataFrame(flat_results)

# --- Save to CSV ---
output_path = '../data/url_analysis_results.csv'
results_df.to_csv(output_path, index=False)

print("Export complete.")
print(f"  File  : {output_path}")
print(f"  Shape : {results_df.shape[0]} rows × {results_df.shape[1]} columns")
print("\nColumns exported:")
for col in results_df.columns:
    print(f"  • {col}")

print("\n============================================================")
print(" Notebook 6 — URL Analysis Engine: COMPLETE")
print("============================================================")
print("  Pipeline steps       : 16")
print("  Signal layers        : Domain (age, HTTPS, TLD, availability)")
print("                         Content (fraud phrases, urgency, contacts)")
print("  Risk score range     : 0 – 100")
print("  Risk levels          : LOW | MEDIUM | HIGH | CRITICAL | INVALID")
print("  Output               : url_analysis_results.csv")
print("============================================================")